# VAE Training for Product Recognition

In this notebook we implemented a **VAE** fine-tuning approach combined with YOLO object detection.

### Components of Pipeline :
1. **VAE Fine-tuning**: Train student-teacher model for feature extraction
2. **YOLO Detection**: Detect products in images
3. **Embedding Extraction**: Extract VAE embeddings for detected crops
4. **FAISS Indexing**: Build efficient similarity search index

### Dataset Used : "https://www.kaggle.com/datasets/diyer22/retail-product-checkout-dataset"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

class RPCVAEDataset(Dataset):
    def __init__(self, root):
        self.root = root
        self.files = [f for f in os.listdir(root) if f.endswith(".jpg")]
        self.transform = T.Compose([
            T.Resize((128, 128)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.1,0.1,0.1,0.02),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        path = os.path.join(self.root, file)
        img = Image.open(path).convert("RGB")
        label = int(file.split("_")[1].split(".")[0])
        return self.transform(img), label

class VAE(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(256*8*8, latent_dim)
        self.fc_logvar = nn.Linear(256*8*8, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 256*8*8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(128,64,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_input(z)
        h = h.view(-1, 256, 8, 8)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + 0.001 * kl_loss

def contrastive_loss(embeddings, labels):
    embeddings = F.normalize(embeddings, dim=1)
    sim = embeddings @ embeddings.T
    labels = labels.unsqueeze(1)
    pos = (labels == labels.T).float()
    neg = (labels != labels.T).float()
    pos_loss = (1 - sim) * pos
    neg_loss = F.relu(sim - 0.3) * neg
    return (pos_loss.sum() + neg_loss.sum()) / (pos.sum() + neg.sum())

def get_embeddings(model, loader, device):
    model.eval()
    embeddings = []
    labels_all = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            mu, _ = model.encode(imgs)
            embeddings.append(mu.cpu())
            labels_all.append(labels)

    return torch.cat(embeddings), torch.cat(labels_all)

device = "cuda" if torch.cuda.is_available() else "cpu"

DATA_PATH = "/kaggle/input/datasets/sarthakd25/cropped-rpc/kaggle/working/cropped_dataset"
CHECKPOINT_PATH = "/kaggle/input/your-checkpoint-path/vae_checkpoint.pth"

dataset = RPCVAEDataset(DATA_PATH)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    drop_last=True
)

model = VAE(latent_dim=256).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint)
    print("Checkpoint loaded successfully")
else:
    print("No checkpoint found, training from scratch")

epochs = 20
lambda_c = 0.5
best_loss = float("inf")

print(f"Starting training for {epochs} epochs")
print(f"Device: {device}")
print(f"Dataset size: {len(dataset)}")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    count = 0

    print(f"\nEpoch {epoch+1}/{epochs} started")

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        if count == 0:
            print(f"Batch shape: {imgs.shape}")

        recon, mu, logvar = model(imgs)

        loss_vae = vae_loss(recon, imgs, mu, logvar)
        loss_con = contrastive_loss(mu, labels)

        loss = loss_vae + lambda_c * loss_con

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

    avg_loss = total_loss / count

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "/kaggle/working/best_vae.pth")
        print(f"Best model updated at epoch {epoch+1}, Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), "/kaggle/working/last_vae.pth")
print("Final model saved")

print("Extracting embeddings...")
embeddings, labels = get_embeddings(model, loader, device)
print(f"Embeddings shape: {embeddings.shape}")

### Setup and Dataset

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.0 MB/s eta 0:00:00:00:0100:01


## Inference Setup
### 1.1 Setup and Dependencies
Import required libraries for model training, data handling, and neural network operations.

In [3]:
import os
import json
import cv2
import torch
import numpy as np
import faiss
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict, Counter
import torch.nn as nn
import torch.nn.functional as F

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.1 MB/s eta 0:00:00a 0:00:01


### 1.2 Define Architecture

In [44]:
class VAE(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.ReLU(),
            nn.Flatten()
        )

        self.fc_mu = nn.Linear(256*8*8, latent_dim)
        self.fc_logvar = nn.Linear(256*8*8, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, 256*8*8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(128,64,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
    
        h = self.decoder_input(z)
        h = h.view(-1, 256, 8, 8)
        recon = self.decoder(h)
    
        return recon, mu, logvar

In [45]:
!pip install torchinfo

from torchinfo import summary

model = VAE(latent_dim=256)

summary(model, input_size=(1, 3, 128, 128))

Layer (type:depth-idx)                   Output Shape              Param #
VAE                                      [1, 3, 128, 128]          --
├─Sequential: 1-1                        [1, 16384]                --
│    └─Conv2d: 2-1                       [1, 32, 64, 64]           1,568
│    └─ReLU: 2-2                         [1, 32, 64, 64]           --
│    └─Conv2d: 2-3                       [1, 64, 32, 32]           32,832
│    └─ReLU: 2-4                         [1, 64, 32, 32]           --
│    └─Conv2d: 2-5                       [1, 128, 16, 16]          131,200
│    └─ReLU: 2-6                         [1, 128, 16, 16]          --
│    └─Conv2d: 2-7                       [1, 256, 8, 8]            524,544
│    └─ReLU: 2-8                         [1, 256, 8, 8]            --
│    └─Flatten: 2-9                      [1, 16384]                --
├─Linear: 1-2                            [1, 256]                  4,194,560
├─Linear: 1-3                            [1, 256]            

In [5]:
pip install torchviz

Note: you may need to restart the kernel to use updated packages.


### 1.3 Environment and Dependencies 

In [7]:
TRAIN_IMG_DIR = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/val2019"
TRAIN_JSON = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_val2019.json"

TEST_IMG_DIR = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/test2019"
TEST_JSON = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_test2019.json"

YOLO_CKPT = "/kaggle/input/datasets/nishant251110053/models/best.pt"
VAE_CKPT = "/kaggle/input/datasets/nishant251110053/models/best_vae.pth"  

FAISS_INDEX_PATH = "product_embeddings.index"
LABEL_MAP_PATH = "embedding_labels.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
print("Loading Models...")

# YOLO
yolo_model = YOLO(YOLO_CKPT)

# VAE
model = VAE(latent_dim=256).to(DEVICE)
model.load_state_dict(torch.load(VAE_CKPT, map_location=DEVICE))
model.eval()

Loading Models...


VAE(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): ReLU()
    (8): Flatten(start_dim=1, end_dim=-1)
  )
  (fc_mu): Linear(in_features=16384, out_features=256, bias=True)
  (fc_logvar): Linear(in_features=16384, out_features=256, bias=True)
  (decoder_input): Linear(in_features=256, out_features=16384, bias=True)
  (decoder): Sequential(
    (0): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): ConvTranspo

### 1.4 Inference and metrics evaluation

In [9]:
transform = T.Compose([
    T.Resize((128, 128)),   
    T.ToTensor()          
])

In [10]:
def get_embedding(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)

    input_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        mu, _ = model.encode(input_tensor)
        emb = mu.cpu().numpy()[0]

    # Normalize for cosine similarity
    return (emb / np.linalg.norm(emb)).astype('float32')

In [11]:
def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)

    boxAArea = (boxA[2]-boxA[0]+1)*(boxA[3]-boxA[1]+1)
    boxBArea = (boxB[2]-boxB[0]+1)*(boxB[3]-boxB[1]+1)

    return interArea / float(boxAArea + boxBArea - interArea)

In [12]:
import matplotlib.pyplot as plt

def compute_metrics(all_scores, all_labels, num_classes):
    precision_curve = []
    recall_curve = []

    thresholds = np.linspace(0, 1, 50)

    for t in thresholds:
        tp, fp, fn = 0, 0, 0

        for scores, labels in zip(all_scores, all_labels):
            preds = [p for p, s in scores if s >= t]

            gt = set(labels)
            pred = set(preds)

            tp += len(gt & pred)
            fp += len(pred - gt)
            fn += len(gt - pred)

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)

        precision_curve.append(precision)
        recall_curve.append(recall)

    return thresholds, precision_curve, recall_curve

In [13]:
def compute_ap(preds, gt, K):
    preds = [p for p, _ in preds]

    gt = set(gt)
    matched = set()

    hits = 0
    score = 0.0

    for i in range(min(K, len(preds))):
        p = preds[i]

        if p in gt and p not in matched:
            hits += 1
            score += hits / (i + 1)
            matched.add(p)

    if len(gt) == 0:
        return 0.0

    return score / len(gt)

In [14]:
def get_ranked_predictions(image, index, db_labels, K=5):
    results = yolo_model(image, verbose=False)

    preds = []

    for r in results:
        for box in r.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)

            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            emb = get_embedding(crop).reshape(1, -1)
            D, I = index.search(emb, K)

            for i, idx in enumerate(I[0]):
                if idx != -1:
                    label = db_labels[idx]
                    score = float(D[0][i])  # similarity score
                    preds.append((label, score))

    # sort by confidence
    preds.sort(key=lambda x: x[1], reverse=True)

    return preds

In [34]:
def build_database():
    print("\n--- Building Database ---")

    with open(TRAIN_JSON) as f:
        data = json.load(f)

    img_id_to_name = {img['id']: img['file_name'] for img in data['images']}
    anns_by_img = defaultdict(list)

    for ann in data['annotations']:
        anns_by_img[ann['image_id']].append(ann)

    all_embeddings = []
    all_labels = []

    image_items = list(img_id_to_name.items())[:6000]

    for img_id, file_name in tqdm(image_items):
        path = os.path.join(TRAIN_IMG_DIR, file_name)
        image = cv2.imread(path)
        if image is None:
            continue

        results = yolo_model(image, verbose=False)
        gt_anns = anns_by_img[img_id]

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)

                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                # Match GT
                best_iou = 0
                best_label = -1

                for ann in gt_anns:
                    gx, gy, gw, gh = ann['bbox']
                    gt_box = [gx, gy, gx+gw, gy+gh]

                    iou = calculate_iou([x1,y1,x2,y2], gt_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_label = ann['category_id']

                if best_iou > 0.6:
                    emb = get_embedding(crop)
                    all_embeddings.append(emb)
                    all_labels.append(int(best_label))

    all_embeddings = np.array(all_embeddings).astype('float32')

    index = faiss.IndexFlatIP(all_embeddings.shape[1])
    index.add(all_embeddings)

    faiss.write_index(index, FAISS_INDEX_PATH)

    with open(LABEL_MAP_PATH, 'w') as f:
        json.dump(all_labels, f)

    print(f"Database size: {len(all_labels)}")

In [46]:
def evaluate():
    print("\n--- Evaluating (Detection mAP) ---")

    import matplotlib.pyplot as plt
    from sklearn.metrics import average_precision_score

    index = faiss.read_index(FAISS_INDEX_PATH)

    with open(LABEL_MAP_PATH) as f:
        db_labels = json.load(f)

    with open(TEST_JSON) as f:
        test_data = json.load(f)

    img_id_to_name = {img['id']: img['file_name'] for img in test_data['images']}

    gt_by_img = defaultdict(list)
    for ann in test_data['annotations']:
        gt_by_img[ann['image_id']].append(ann)

    image_items = list(img_id_to_name.items())[:6000]

    # knn hyperparameter
    K = 5
    
    iou_thresholds = np.arange(0.5, 1.0, 0.05)

    all_gt_labels_dict = {t: [] for t in iou_thresholds}
    all_scores_dict = {t: [] for t in iou_thresholds}

    total_gt = 0
    tp_50 = 0

    all_gt_labels = []
    all_scores = []

    #  Main Loop
    for img_id, file_name in tqdm(image_items):

        path = os.path.join(TEST_IMG_DIR, file_name)
        image = cv2.imread(path)
        if image is None:
            continue

        results = yolo_model(image, verbose=False)
        gt_anns = gt_by_img[img_id]

        total_gt += len(gt_anns)

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy():

                x1, y1, x2, y2 = map(int, box)

                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                emb = get_embedding(crop).reshape(1, -1)
                D, I = index.search(emb, K)

                neighbor_labels = [db_labels[idx] for idx in I[0] if idx != -1]

                if neighbor_labels:
                    pred_cls = Counter(neighbor_labels).most_common(1)[0][0]
                    score = float(D[0][0])

                    
                    best_iou = 0
                    matched_gt_cls = -1

                    for ann in gt_anns:
                        gx, gy, gw, gh = ann['bbox']
                        gt_box = [gx, gy, gx+gw, gy+gh]

                        iou = calculate_iou([x1,y1,x2,y2], gt_box)

                        if iou > best_iou:
                            best_iou = iou
                            matched_gt_cls = ann['category_id']

                    
                    for t in iou_thresholds:
                        is_tp = (best_iou >= t and pred_cls == matched_gt_cls)

                        if t == 0.5 and is_tp:
                            tp_50 += 1

                        all_gt_labels_dict[t].append(1 if is_tp else 0)
                        all_scores_dict[t].append(score)

                    is_tp_50 = (best_iou >= 0.5 and pred_cls == matched_gt_cls)
                    all_gt_labels.append(1 if is_tp_50 else 0)
                    all_scores.append(score)

    
    # FINAL METRICS
    precision = np.mean(all_gt_labels) if all_gt_labels else 0
    recall = tp_50 / total_gt if total_gt > 0 else 0

    # mAP50 + mAP50-95
    map_list = []

    for t in iou_thresholds:
        labels = all_gt_labels_dict[t]
        scores = all_scores_dict[t]

        if len(labels) > 0:
            ap = average_precision_score(labels, scores)
            map_list.append(ap)

    map50 = map_list[0] if len(map_list) > 0 else 0
    map50_95 = np.mean(map_list) if len(map_list) > 0 else 0

    print("\n====== FINAL RESULTS ======")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"mAP50: {map50:.4f}")

    thresholds = np.linspace(min(all_scores), max(all_scores), 50) if all_scores else [0]

    precisions = []
    recalls = []

    for t in thresholds:
        tp, fp, fn = 0, 0, 0

        for label, score in zip(all_gt_labels, all_scores):
            if score >= t:
                if label == 1:
                    tp += 1
                else:
                    fp += 1
            else:
                if label == 1:
                    fn += 1

        precision_t = tp / (tp + fp + 1e-6)
        recall_t = tp / (tp + fn + 1e-6)

        precisions.append(precision_t)
        recalls.append(recall_t)

    
    plt.figure()
    plt.plot(recalls, precisions)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("PR Curve (Detection)")
    plt.savefig("/kaggle/working/pr_curve.png")

    plt.figure()
    plt.plot(thresholds, precisions)
    plt.xlabel("Confidence")
    plt.ylabel("Precision")
    plt.title("Precision-Confidence Curve")
    plt.savefig("/kaggle/working/precision_conf_curve.png")

    f1_scores = [2*p*r/(p+r+1e-6) for p, r in zip(precisions, recalls)]

    plt.figure()
    plt.plot(thresholds, f1_scores)
    plt.xlabel("Confidence")
    plt.ylabel("F1")
    plt.title("F1-Confidence Curve")
    plt.savefig("/kaggle/working/f1_curve.png")

    print(" Curves saved in /kaggle/working/")

In [17]:
def save_predictions(num_images=12):
    print("\n--- Saving Predictions Grid (Random Images) ---")

    import random

    index = faiss.read_index(FAISS_INDEX_PATH)

    with open(LABEL_MAP_PATH) as f:
        db_labels = json.load(f)

    with open(TRAIN_JSON) as f:
        test_data = json.load(f)

    category_map = {
        cat["id"]: f"{cat['id']}_{cat['supercategory']}"
        for cat in test_data["categories"]
    }

    img_id_to_name = {img['id']: img['file_name'] for img in test_data['images']}

    image_items = random.sample(
        list(img_id_to_name.items()),
        min(num_images, len(img_id_to_name))
    )

    images_out = []

    for i, (img_id, file_name) in enumerate(image_items):
        path = os.path.join(TRAIN_IMG_DIR, file_name)
        image = cv2.imread(path)
        if image is None:
            continue

        results = yolo_model(image, verbose=False)

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy():

                x1, y1, x2, y2 = map(int, box)
                crop = image[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                emb = get_embedding(crop).reshape(1, -1)
                D, I = index.search(emb, 5)

                neighbor_labels = [db_labels[idx] for idx in I[0] if idx != -1]

                if neighbor_labels:
                    pred_id = Counter(neighbor_labels).most_common(1)[0][0]

                    # label: id_supercategory + confidence
                    conf = float(D[0][0])
                    label = f"{category_map.get(pred_id, pred_id)} {conf:.2f}"

                    
                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)

                    
                    (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)

                    cv2.rectangle(image, (x1, y1 - h - 10), (x1 + w, y1), (0, 0, 255), -1)

                    cv2.putText(image, label, (x1, y1 - 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        images_out.append(image)

    max_h = 1080
    max_w = 1080

    resized_images = [
        cv2.resize(img, (max_w, max_h)) for img in images_out
    ]

   
    rows = int(np.ceil(np.sqrt(len(resized_images))))
    cols = rows

    grid = np.zeros((rows * max_h, cols * max_w, 3), dtype=np.uint8)

    for idx, img in enumerate(resized_images):
        r = idx // cols
        c = idx % cols
        grid[r*max_h:(r+1)*max_h, c*max_w:(c+1)*max_w] = img

    save_path = "/kaggle/working/pred_grid.jpg"
    cv2.imwrite(save_path, grid)

    print(f" Saved grid: {save_path}")

In [37]:
def evaluate_rpc_metrics():
    print("\n===== RPC METRICS (HARD IMAGES) =====")

    index = faiss.read_index(FAISS_INDEX_PATH)
    with open(LABEL_MAP_PATH, 'r') as f:
        db_labels = json.load(f)

    with open(TEST_JSON) as f:
        test_data = json.load(f)

    img_map = {img['id']: img for img in test_data['images']}

    ann_by_img = defaultdict(list)
    for ann in test_data['annotations']:
        ann_by_img[ann['image_id']].append(ann)

    #  Filter HARD images
    hard_ids = [
        img_id for img_id, img in img_map.items()
        if img.get("level") == "hard" or img.get("difficulty") == "hard"
    ]
    
    # LIMIT TO FIRST 8000
    hard_ids = hard_ids[:8000]
    
    print("Total HARD images:", len(hard_ids))

    # METRIC STORAGE
    yolo_cAcc = 0
    vae_cAcc = 0

    yolo_acd, vae_acd = [], []
    yolo_mccd, vae_mccd = [], []
    yolo_mciou, vae_mciou = [], []

    total_images = 0

    for img_id in tqdm(hard_ids):

        file_name = img_map[img_id]['file_name']
        img_path = os.path.join(TEST_IMG_DIR, file_name)

        image = cv2.imread(img_path)
        if image is None:
            continue

        gt_anns = ann_by_img[img_id]

        # GT COUNT 
        gt_counts = Counter([ann['category_id'] for ann in gt_anns])

     
        # YOLO PREDICTION
        results = yolo_model(image, verbose=False)

        yolo_labels = []
        for r in results:
            if r.boxes.cls is not None:
                yolo_labels.extend(r.boxes.cls.cpu().numpy().astype(int))

        yolo_counts = Counter(yolo_labels)

        # vae PREDICTION
        
        vae_labels = []

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)

                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                emb = get_embedding(crop).reshape(1, -1)
                D, I = index.search(emb, 5)

                labels = [db_labels[i] for i in I[0] if i != -1]

                if labels:
                    pred = Counter(labels).most_common(1)[0][0]
                    vae_labels.append(pred)

        vae_counts = Counter(vae_labels)

        # METRICS
        
        d_acd, d_mccd, d_mciou = compute_rpc_metrics(gt_counts, vae_counts)

        vae_acd.append(d_acd)
        vae_mccd.append(d_mccd)
        vae_mciou.append(d_mciou)

        

        if gt_counts == vae_counts:
            vae_cAcc += 1

        total_images += 1

    # FINAL RESULTS
    print("\n===== FINAL RESULTS =====")

    print("\n--- vae ---")
    print(f"ACD   : {np.mean(vae_acd):.2f}")
    print(f"mCCD  : {np.mean(vae_mccd):.2f}")
    print(f"mCIoU : {100 * np.mean(vae_mciou):.2f}%")

In [38]:
from collections import defaultdict, Counter
import numpy as np

def compute_rpc_metrics(gt_counts, pred_counts):

    categories = set(gt_counts.keys()) | set(pred_counts.keys())

    # ===== ACD =====
    total_gt = sum(gt_counts.values())
    total_pred = sum(pred_counts.values())
    acd = abs(total_gt - total_pred)

    # ===== mCCD =====
    mccd_list = []
    for c in categories:
        g = gt_counts.get(c, 0)
        p = pred_counts.get(c, 0)

        if g > 0:
            mccd_list.append(abs(p - g) / g)

    mccd = np.mean(mccd_list) if mccd_list else 0

    # ===== mCIoU =====
    inter = 0
    union = 0
    for c in categories:
        g = gt_counts.get(c, 0)
        p = pred_counts.get(c, 0)

        inter += min(g, p)
        union += max(g, p)

    mciou = inter / union if union > 0 else 0

    return acd, mccd, mciou

In [4]:
evaluate_rpc_metrics()

In [3]:
if __name__ == "__main__":
    build_database()
    evaluate()
    save_predictions(12)